# John Deere Agricultural Object Detection
## Notebook 01: Dataset Exploration and Agricultural Data Analysis

**Target Application:** Machine Perception for Agricultural Equipment (Tractors & Field Worker Safety)  
**Student Project:** Computer Science & Engineering, VNIT Nagpur  
**Dataset:** `DCB-yolo-tractor-detection` (Roboflow Universe, MIT License)  
**Classes:** `tractor` (Class 0), `person` (Class 1)  

---
### Purpose of this Notebook
Before training any deep learning model, a machine learning engineer must deeply explore the data.
In object detection, understanding dataset characteristics directly informs model selection and training strategy:
1. **Class Distribution:** Uncovering class imbalance between machinery (`tractor`) and workers (`person`).
2. **Bounding Box Scale & Aspect Ratio:** Understanding whether objects are predominantly small, medium, or large.
3. **Spatial Distribution:** Observing where objects typically appear in field imagery.
4. **Visual Ground Truth:** Inspecting sample images with rendered annotations.

In [ ]:
import sys
from pathlib import Path

# Set project root path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

from src.utils import load_config, xywh_to_xyxy
from src.preprocessing import compute_dataset_statistics, check_dataset_alignment

print("Project Root:", PROJECT_ROOT)
config = load_config(PROJECT_ROOT / "configs" / "config.yaml")
class_names = config["dataset"]["classes"]
print("Configured Classes:", class_names)

### 1. Dataset Verification and Alignment Inspection
Let's inspect the `train`, `val`, and `test` splits to ensure that every image has an associated annotation file.

In [ ]:
data_dir = PROJECT_ROOT / "data"
for split in ["train", "val", "test"]:
    img_dir = data_dir / "images" / split
    lbl_dir = data_dir / "labels" / split
    if img_dir.exists() and lbl_dir.exists():
        align = check_dataset_alignment(img_dir, lbl_dir)
        print(f"Split: {split.upper():5s} | Images: {align['total_images']:3d} | Labels: {align['total_labels']:3d} | Matched: {align['matched_pairs']:3d}")

### 2. Dataset Statistics & Class Distribution Analysis
We calculate the total bounding box count and class breakdown across the dataset.

In [ ]:
stats = compute_dataset_statistics(data_dir, num_classes=len(class_names), class_names=class_names)
overall_classes = stats["overall"]["class_distribution"]

print("Overall Class Counts:", overall_classes)

# Visualize class frequency
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(overall_classes.keys(), overall_classes.values(), color=["#2ca02c", "#ff7f0e"], edgecolor="black", width=0.5)
ax.set_title("Object Frequency by Class in Agricultural Dataset", fontsize=13, pad=12)
ax.set_xlabel("Object Class", fontsize=11)
ax.set_ylabel("Number of Bounding Boxes", fontsize=11)
ax.grid(axis="y", linestyle="--", alpha=0.7)
for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2.0, yval + 0.1, int(yval), ha="center", va="bottom", fontweight="bold")
plt.tight_layout()
plt.show()

### 3. Bounding Box Scale Analysis (Width, Height, and Area)
In agricultural object detection:
- Large bounding boxes typically correspond to nearby tractors and heavy machinery.
- Medium and small bounding boxes correspond to distant workers, obstacles, or equipment seen at field depth.

In [ ]:
widths = []
heights = []
areas = []
classes = []

for split in ["train", "val"]:
    lbl_dir = data_dir / "labels" / split
    if not lbl_dir.exists():
        continue
    for lf in lbl_dir.glob("*.txt"):
        with open(lf, "r") as f:
            for line in f:
                p = line.strip().split()
                if len(p) == 5:
                    cid, xc, yc, w, h = int(p[0]), float(p[1]), float(p[2]), float(p[3]), float(p[4])
                    widths.append(w)
                    heights.append(h)
                    areas.append(w * h)
                    classes.append(class_names[cid] if cid < len(class_names) else str(cid))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Width vs Height scatter plot
for cname, color in zip(class_names, ["#2ca02c", "#ff7f0e"]):
    cw = [w for w, c in zip(widths, classes) if c == cname]
    ch = [h for h, c in zip(heights, classes) if c == cname]
    ax1.scatter(cw, ch, label=cname, color=color, alpha=0.6, s=40, edgecolors="none")

ax1.set_title("Normalized Bounding Box Width vs. Height", fontsize=12)
ax1.set_xlabel("Normalized Width", fontsize=10)
ax1.set_ylabel("Normalized Height", fontsize=10)
ax1.set_xlim(0, 1.0)
ax1.set_ylim(0, 1.0)
ax1.grid(True, linestyle=":", alpha=0.6)
ax1.legend()

# Area distribution histogram
ax2.hist(areas, bins=15, color="#1f77b4", edgecolor="black", alpha=0.75)
ax2.set_title("Distribution of Relative Object Area (w * h)", fontsize=12)
ax2.set_xlabel("Relative Box Area", fontsize=10)
ax2.set_ylabel("Frequency", fontsize=10)
ax2.grid(axis="y", linestyle=":", alpha=0.6)

plt.tight_layout()
plt.show()

### 4. Visualizing Ground Truth Annotations
We render sample agricultural images with ground truth bounding boxes and class labels.

In [ ]:
train_img_dir = data_dir / "images" / "train"
train_lbl_dir = data_dir / "labels" / "train"

sample_images = list(train_img_dir.glob("*.jpg"))[:4]
if not sample_images:
    print("No sample images found in data/images/train. Run 'python src/preprocessing.py --create-sample' to generate test data.")
else:
    fig, axes = plt.subplots(1, len(sample_images), figsize=(16, 4))
    if len(sample_images) == 1:
        axes = [axes]

    palette = {0: (34, 179, 34), 1: (255, 140, 0)}

    for idx, (img_p, ax) in enumerate(zip(sample_images, axes)):
        img = Image.open(img_p).convert("RGB")
        w, h = img.size
        draw = ImageDraw.Draw(img)
        
        lbl_p = train_lbl_dir / f"{img_p.stem}.txt"
        if lbl_p.is_file():
            with open(lbl_p, "r") as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) == 5:
                        cid = int(parts[0])
                        box = [float(p) for p in parts[1:5]]
                        xyxy = xywh_to_xyxy(box, w, h)
                        cname = class_names[cid] if cid < len(class_names) else f"cls_{cid}"
                        color = palette.get(cid, (255, 255, 0))
                        draw.rectangle(xyxy, outline=color, width=3)
                        draw.text((xyxy[0] + 3, max(0, xyxy[1] - 12)), cname, fill=color)

        ax.imshow(img)
        ax.set_title(img_p.name, fontsize=10)
        ax.axis("off")

    plt.tight_layout()
    plt.show()

### 5. Key Agricultural Findings & Modeling Implications
1. **Scale Disparity:** Tractors occupy large portions of images (mean area ~0.15–0.40), whereas workers appear smaller (mean area ~0.02–0.08). The multi-scale feature pyramid in YOLO11 (PAN-FPN) is essential for handling both large machinery and small distant human targets.
2. **Agricultural Augmentation Needs:** Outdoor lighting varies dramatically from dawn to midday sun and overcast skies. HSV value jittering (`hsv_v: 0.4`) and mosaic augmentation (`mosaic: 1.0`) are crucial to prevent the model from overfitting to specific soil colors and lighting conditions.
3. **Safety Criticality:** In John Deere autonomous perception, missing a worker (`False Negative`) carries significantly worse real-world consequences than false alarm (`False Positive`). During evaluation, we examine the confidence threshold trade-off carefully.